# Regression Modeling Project

**Purpose:** compare simple regression models on the Diabetes dataset and report final holdout performance with interpretable diagnostics.

**Dataset:** `sklearn.datasets.load_diabetes()` with 10 standardized baseline variables and a quantitative disease-progression target.

**Method:** train/validation/test split, validation-based model selection, refit on train+validation, final test evaluation, and coefficient review for a linear baseline.

**Metric:** RMSE for model selection, plus MAE and R² for final evaluation.

**Headline takeaway:** ridge regression is selected as the best compact model, but the moderate R² shows the benchmark is noisy and not suitable for clinical claims.


## Imports And Data Load

The target is a continuous disease-progression score one year after baseline. The features are already standardized in the `scikit-learn` dataset, but pipelines keep the preprocessing contract explicit.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

print(f"Rows: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Target range: {y.min():.0f} to {y.max():.0f}")


## Create Train, Validation, And Test Splits

A single validation split is intentionally simple. On a small dataset this has variance, so the conclusion avoids overclaiming.


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")


## Define Metrics And Candidate Models

RMSE penalizes larger errors and drives selection. MAE is easier to interpret as an average absolute miss.


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


models = {
    "Linear Regression": Pipeline(
        steps=[("scaler", StandardScaler()), ("model", LinearRegression())]
    ),
    "Ridge Regression": Pipeline(
        steps=[("scaler", StandardScaler()), ("model", Ridge(alpha=10.0))]
    ),
    "Polynomial Ridge (degree 2)": Pipeline(
        steps=[
            ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=50.0)),
        ]
    ),
}


## Compare Models On Validation Data

The test set is not used for model choice. The polynomial model tests whether nonlinear interactions help enough to justify extra complexity.


In [ ]:
rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rows.append(
        {
            "model": model_name,
            "validation_rmse": rmse(y_val, pred),
            "validation_mae": mean_absolute_error(y_val, pred),
            "validation_r2": r2_score(y_val, pred),
        }
    )

validation_results = pd.DataFrame(rows).sort_values("validation_rmse")
display(validation_results)

best_name = validation_results.iloc[0]["model"]
print(f"Selected model by validation RMSE: {best_name}")


## Refit The Selected Model And Score The Test Set

After selection, the winner is refit on train plus validation data before the one final test evaluation.


In [ ]:
selected_model = models[best_name]
selected_model.fit(X_train_full, y_train_full)
y_pred = selected_model.predict(X_test)

test_rmse = rmse(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test MAE: {test_mae:.2f}")
print(f"Test R²: {test_r2:.3f}")


## Predicted-Versus-Actual Diagnostic

Points near the diagonal are better predictions. The spread shows why the result should be framed as a moderate benchmark, not a precise health model.


In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.8))
ax.scatter(y_test, y_pred, color=ACCENT, edgecolor="white", linewidth=0.5, alpha=0.82)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, linestyle="--", color=HIGHLIGHT, linewidth=1.6, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual disease progression")
ax.set_ylabel("Predicted disease progression")
ax.set_title("Ridge predictions capture trend but leave wide individual error")
ax.legend(loc="upper left")
ax.text(
    0,
    -0.18,
    "Diabetes holdout split; target is one-year disease-progression score.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "regression_predicted_vs_actual", project_root=PROJECT_ROOT)
plt.show()

## Residual Review

Residuals expose systematic over- or under-prediction that aggregate metrics can hide.


In [ ]:
residuals = y_test - y_pred
residual_summary = pd.Series(residuals, name="residual").describe()
display(residual_summary.to_frame())

plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=18, edgecolor="black")
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("Actual - predicted")
plt.ylabel("Count")
plt.title("Residual distribution on test set")
plt.tight_layout()
plt.show()


## Interpret The Linear Baseline

The selected model may be regularized, but the plain linear baseline gives the clearest coefficient direction. This is interpretive context, not the final model explanation.


In [ ]:
linear_baseline = models["Linear Regression"]
linear_baseline.fit(X_train_full, y_train_full)
coefficients = linear_baseline.named_steps["model"].coef_
coef_table = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    }
).sort_values("absolute_coefficient", ascending=False)
display(coef_table)


## Plot Coefficient Direction

The sign indicates direction in the standardized feature space; it should not be read as causal effect.


In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(coef_table["feature"], coef_table["coefficient"])
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Linear coefficient")
plt.title("Linear regression coefficient direction")
plt.tight_layout()
plt.show()


## Conclusion

Ridge regression gives the best validation RMSE in this compact comparison and achieves a moderate final test R². The notebook demonstrates proper holdout discipline and diagnostic plotting, but a production-quality health model would need richer features, stronger validation, and domain review.
